# DX 704 Week 9 Project

This week's project will build an email spam classifier based on the Enron email data set.
You will perform your own feature extraction, and use naive Bayes to estimate the probability that a particular email is spam or not.
Finally, you will review the tradeoffs from different thresholds for automatically sending emails to the junk folder.

The full project description and a template notebook are available on GitHub: [Project 9 Materials](https://github.com/bu-cds-dx704/dx704-project-09).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [8]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

zsh:1: command not found: wget


In [7]:
import pandas as pd

In [10]:
from pathlib import Path
from urllib.request import urlretrieve
import pandas as pd

data_path = Path.cwd() / "enron_spam_data.zip"
data_url = (
    "https://raw.githubusercontent.com/MWiechmann/"
    "enron_spam_data/refs/heads/master/enron_spam_data.zip"
)

if not data_path.exists():
    urlretrieve(data_url, data_path)

enron_spam_data = pd.read_csv(data_path)
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [11]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Design a Feature Extractor

Design a feature extractor for this data set and write out two files of features based on the text.
Don't forget that both the Subject and Message columns are relevant sources of text data.
For each email, you should count the number of repetitions of each feature present.
The auto-grader will assume that you are using a multinomial distribution in the following problems.

In [12]:
# YOUR CHANGES HERE

import re
from collections import Counter

def extract_features(subject, message):
    text = f"{subject} {message}".lower()
    tokens = re.findall(r"\b[a-z0-9]+\b", text)
    return dict(Counter(tokens))

features = enron_spam_data.apply(
    lambda row: extract_features(
        "" if pd.isna(row["Subject"]) else row["Subject"],
        "" if pd.isna(row["Message"]) else row["Message"],
    ),
    axis=1,
)

Assign a row to the test data set if `Message ID % 30 == 0` and assign it to the training data set otherwise.
Write two files, "train-features.tsv" and "test-features.tsv" with two columns, Message ID and features_json.
The features_json column should contain a JSON dictionary where the keys are your feature names and the values are integer feature values.
This will give us a sparse feature representation.


In [13]:
# YOUR CHANGES HERE

import json

feature_data = enron_spam_data[["Message ID"]].copy()
feature_data["features_json"] = features.map(json.dumps)

test_mask = feature_data["Message ID"] % 30 == 0

train_features = feature_data.loc[~test_mask].copy()
test_features = feature_data.loc[test_mask].copy()

train_features.to_csv("train-features.tsv", sep="\t", index=False)
test_features.to_csv("test-features.tsv", sep="\t", index=False)

print(f"Training emails: {len(train_features)}")
print(f"Test emails: {len(test_features)}")

Training emails: 32592
Test emails: 1124


Submit "train-features.tsv" and "test-features.tsv" in Gradescope.

Hint: these features will be graded based on the test accuracy of a logistic regression based on the training features.
This is to make sure that your feature set is not degenerate; you do not need to compute this regression yourself.
You can separately assess your feature quality based on your results in part 6.

## Part 3: Compute Conditional Probabilities

Based on your training data, compute appropriate conditional probabilities for use with naïve Bayes.
Use of additive smoothing with $\alpha=1$ to avoid zeros.


In [14]:
# YOUR CHANGES HERE

train_mask = enron_spam_data["Message ID"] % 30 != 0

ham_counts = Counter()
spam_counts = Counter()

for idx, row in enron_spam_data.loc[train_mask].iterrows():
    if row["Spam/Ham"] == "spam":
        spam_counts.update(features.loc[idx])
    else:
        ham_counts.update(features.loc[idx])

vocabulary = sorted(set(ham_counts) | set(spam_counts))
vocab_size = len(vocabulary)

ham_total = sum(ham_counts.values())
spam_total = sum(spam_counts.values())

# Additive smoothing with alpha = 1
feature_probabilities = pd.DataFrame({
    "feature": vocabulary,
    "ham_probability": [
        (ham_counts[word] + 1) / (ham_total + vocab_size)
        for word in vocabulary
    ],
    "spam_probability": [
        (spam_counts[word] + 1) / (spam_total + vocab_size)
        for word in vocabulary
    ],
})

feature_probabilities.head()

,feature,ham_probability,spam_probability
0,0,0.000953,0.001768
1,00,0.001460,0.001639
2,000,0.001006,0.001478
3,0000,0.000007,0.000137
4,000000,0.000001,0.000042


Save the conditional probabilities in a file "feature-probabilities.tsv" with columns feature, ham_probability and spam_probability.

In [15]:
# YOUR CHANGES HERE

feature_probabilities.to_csv(
    "feature-probabilities.tsv",
    sep="\t",
    index=False,
)

Submit "feature-probabilities.tsv" in Gradescope.

## Part 4: Implement a Naïve Bayes Classifier

Implement a naïve Bayes classifier based on your previous feature probabilities.

In [16]:
# YOUR CHANGES HERE

import numpy as np

# Estimate class priors using only the training emails.
train_labels = enron_spam_data.loc[train_mask, "Spam/Ham"]
spam_prior = (train_labels == "spam").mean()
ham_prior = (train_labels == "ham").mean()

log_feature_probs = {
    row["feature"]: (
        np.log(row["ham_probability"]),
        np.log(row["spam_probability"]),
    )
    for _, row in feature_probabilities.iterrows()
}

def predict_probabilities(email_features):
    log_ham = np.log(ham_prior)
    log_spam = np.log(spam_prior)

    for word, count in email_features.items():
        if word in log_feature_probs:
            ham_prob, spam_prob = log_feature_probs[word]
            log_ham += count * ham_prob
            log_spam += count * spam_prob

    # Normalize in log space to avoid numerical underflow.
    log_total = np.logaddexp(log_ham, log_spam)

    return {
        "ham": float(np.exp(log_ham - log_total)),
        "spam": float(np.exp(log_spam - log_total)),
    }

Save your prediction probabilities to "train-predictions.tsv" with columns Message ID, ham and spam.

In [17]:
# YOUR CHANGES HERE

train_predictions = enron_spam_data.loc[
    train_mask, ["Message ID"]
].copy()

probabilities = pd.DataFrame(
    [
        predict_probabilities(email_features)
        for email_features in features.loc[train_mask]
    ],
    index=train_predictions.index,
)

train_predictions[["ham", "spam"]] = probabilities[["ham", "spam"]]

train_predictions.to_csv(
    "train-predictions.tsv",
    sep="\t",
    index=False,
)

train_predictions.head()

,Message ID,ham,spam
1,1,1.0,1.566871e-186
2,2,1.0,1.520225e-12
3,3,1.0,7.126796e-155
4,4,1.0,2.713786e-150
5,5,1.0,5.918101e-40


Submit "train-predictions.tsv" in Gradescope.

## Part 5: Predict Spam Probability for Test Data

Use your previous classifier to predict spam probability for the test data.
If the test data has tokens that you did not model, just ignore them.

In [18]:
# YOUR CHANGES HERE

test_mask = enron_spam_data["Message ID"] % 30 == 0

test_predictions = enron_spam_data.loc[
    test_mask, ["Message ID"]
].copy()

probabilities = pd.DataFrame(
    [
        predict_probabilities(email_features)
        for email_features in features.loc[test_mask]
    ],
    index=test_predictions.index,
)

test_predictions[["ham", "spam"]] = probabilities[["ham", "spam"]]

test_predictions.head()

,Message ID,ham,spam
0,0,0.054552,9.454476e-01
30,30,1.000000,1.159190e-84
60,60,1.000000,1.180465e-12
90,90,1.000000,7.572908e-34
120,120,1.000000,2.341464e-188


Save your prediction probabilities in "test-predictions.tsv" with the same columns as "train-predictions.tsv".

In [19]:
# YOUR CHANGES HERE

test_predictions.to_csv(
    "test-predictions.tsv",
    sep="\t",
    index=False,
)

Submit "test-predictions.tsv" in Gradescope.

## Part 6: Construct ROC Curve

For every probability threshold from 0.01 to .99 in increments of 0.01, compute the false and true positive rates from the test data using the spam class for positives.
That is, if the predicted spam probability is greater than or equal to the threshold, predict spam.

In [20]:
# YOUR CHANGES HERE

actual_spam = (
    enron_spam_data.loc[test_predictions.index, "Spam/Ham"] == "spam"
)
actual_ham = ~actual_spam

roc_rows = []

for threshold in np.arange(1, 100) / 100:
    predicted_spam = test_predictions["spam"] >= threshold

    false_positive_rate = (
        (predicted_spam & actual_ham).sum() / actual_ham.sum()
    )
    true_positive_rate = (
        (predicted_spam & actual_spam).sum() / actual_spam.sum()
    )

    roc_rows.append({
        "threshold": threshold,
        "false_positive_rate": false_positive_rate,
        "true_positive rate": true_positive_rate,
    })

roc = pd.DataFrame(roc_rows)
roc.head()

,threshold,false_positive_rate,true_positive rate
0,0.01,0.034420,0.996503
1,0.02,0.034420,0.996503
2,0.03,0.032609,0.996503
3,0.04,0.032609,0.996503
4,0.05,0.028986,0.996503


Save this data in a file "roc.tsv" with columns threshold, false_positive_rate and true_positive rate.

In [21]:
# YOUR CHANGES HERE

roc.to_csv("roc.tsv", sep="\t", index=False)

Submit "roc.tsv" in Gradescope.

## Part 7: OpenAI API Key

Sign up for an OpenAI API acount at https://platform.openai.com/.
If you already have an OpenAI account (e.g. for ChatGPT), you may reuse that account.

Create a "dx704" project at https://platform.openai.com/settings/organization/projects, setup a $10/month spend limit, and create an API key for the project at https://platform.openai.com/api-keys.
You will use this API key for projects in later weeks.


No submission needed.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.